# Pré-traduction des fiches FR → wolof (NLLB-200)
 
> **Objectif :** premier pas du swap wolof — traduire les fiches RAG existantes du français vers le wolof avec NLLB-200, poser les limites connues du modèle, préparer le terrain pour l'indexation ChromaDB wolof (J2).

## Setup

In [ ]:
import os
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    PROJECT_ROOT = Path("/content/noo-far-pipeline")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/noofar-ia/noo-far-pipeline.git /content/noo-far-pipeline
    else:
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

import sys
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projet : {PROJECT_ROOT} | Sur Colab : {ON_COLAB}")

## Chargement de NLLB


In [ ]:
from huggingface_hub import login
login(token="Mon_TOken")  # ne jamais committer avec la vraie valeur

In [ ]:
fiches_fr_dir = PROJECT_ROOT / "rag" / "fiches" / "fr"
fiches = sorted(fiches_fr_dir.glob("*.md"))
print(f"{len(fiches)} fiches trouvées : {[f.name for f in fiches]}")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"NLLB-200 chargé sur {device}")

## Traduction des fiches en wolof

Fonction de traduction, phrase par phrase ou passage par passage

Traduire un texte long d'un coup peut dégrader la qualité — découper par paragraphes (déjà naturellement séparés par les ## markdown) donne de meilleurs résultats.

In [ ]:
#Fonction de traduction
def translate_nllb(text, src_lang="fra_Latn", tgt_lang="wol_Latn", max_length=200):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        no_repeat_ngram_size=3,   # empêche la répétition de séquences de 3 tokens
        repetition_penalty=1.3,    # pénalise la réutilisation de tokens déjà générés
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

In [ ]:
def translate_fiche(fiche_path):
    """Traduit une fiche markdown, en gardant la structure (titres, sections)."""
    text = fiche_path.read_text(encoding="utf-8")
    lignes = text.split("\n")

    lignes_traduites = []
    for ligne in lignes:
        if not ligne.strip():
            lignes_traduites.append("")
            continue
        if ligne.startswith("#"):
            # Titre : traduire le texte, garder le niveau de titre (#, ##...)
            niveau = len(ligne) - len(ligne.lstrip("#"))
            titre_texte = ligne.lstrip("#").strip()
            titre_traduit = translate_nllb(titre_texte)
            lignes_traduites.append("#" * niveau + " " + titre_traduit)
        elif ligne.startswith("*Sources"):
            # Ne pas traduire la ligne de source/citation — garder telle quelle
            lignes_traduites.append(ligne)
        else:
            lignes_traduites.append(translate_nllb(ligne))

    return "\n".join(lignes_traduites)

In [ ]:
fiches_wo_dir = PROJECT_ROOT / "rag" / "fiches" / "wo"
fiches_wo_dir.mkdir(parents=True, exist_ok=True)

for fiche_path in fiches:
    print(f"Traduction : {fiche_path.name}...")
    texte_traduit = translate_fiche(fiche_path)
    output_path = fiches_wo_dir / fiche_path.name
    output_path.write_text(texte_traduit, encoding="utf-8")
    print(f"  → {output_path}")

In [ ]:
for f in sorted(fiches_wo_dir.glob("*.md")):
    print(f"=== {f.name} ===")
    print(f.read_text(encoding="utf-8"))
    print()

# Limites observées — NLLB-200 (fr → wolof), S2-J1

> **Contexte.** Synthèse des artefacts et limites rencontrés en traduisant les 6 fiches RAG françaises vers le wolof avec `facebook/nllb-200-distilled-600M`, le 28 juillet 2026 (S2-J1). À compléter au fil de la relecture des fiches restantes.

---

## 1. Hallucination répétitive (boucle sur un mot)

**Observation.** Sur certaines lignes sources, le modèle génère un mot puis reste bloqué à le répéter jusqu'à `max_length`, sans jamais varier.

**Exemple concret :**
```
Ndàllub àllug àllug àllug àllug àllug àllug àllug àllug àllug àllug àllug àllug...
[répété jusqu'à la limite de longueur]
```

**Cause probable.** Pattern connu des modèles de génération séquentielle face à une source difficile à traduire (vocabulaire rare, structure complexe) — le wolof étant une langue à très faibles ressources pour NLLB, le modèle "décroche" statistiquement plus facilement que sur des langues mieux représentées dans son entraînement.

**Correctif appliqué.** Ajout de `no_repeat_ngram_size=3` et `repetition_penalty=1.3` dans l'appel à `model.generate()` — empêche mécaniquement la répétition de séquences de 3 tokens consécutifs et pénalise la réutilisation de tokens déjà générés.

**Statut.** Corrigé au niveau du code. Prévoir malgré tout une vérification systématique (recherche de mots consécutifs identiques) sur chaque fiche traduite, le correctif réduisant le risque sans l'éliminer totalement.

---

## 2. Hallucination sur vocabulaire technique agricole rare

**Observation.** Un terme technique peu représenté dans l'entraînement de NLLB est confondu avec un mot français phonétiquement proche, mais sémantiquement sans rapport.

**Exemple concret :**
```
Source (fr) : "Bourgou" (fourrage local, plante utilisée en élevage sahélien)
Traduction obtenue : "Bourgogne" (région viticole française)
```

**Cause probable.** "Bourgou" n'est vraisemblablement pas — ou très peu — présent dans le corpus d'entraînement de NLLB en wolof. Le modèle rapproche le mot inconnu d'un mot français qu'il connaît bien et qui lui ressemble phonétiquement, plutôt que de le laisser tel quel ou de signaler l'incertitude.

**Portée.** Pattern à surveiller particulièrement sur le vocabulaire spécifique au Sahel (noms de plantes locales, pratiques d'élevage traditionnelles, unités ou dénominations non standard) — probablement sous-représenté dans les corpus d'entraînement généralistes de NLLB, orientés vers des domaines plus courants (actualité, littérature, contenu web général).

**Statut.** Non corrigeable automatiquement — nécessite une relecture manuelle ciblée sur les termes techniques/noms propres de chaque fiche. Ce cas précis (Bourgou/Bourgogne) à corriger manuellement dans `rag/fiches/wo/alimentation.md`.

---

## 3. Mélange français/wolof au sein d'une même traduction

**Observation.** Certains passages traduits mélangent des mots français non traduits avec du wolof, produisant un résultat hybride incohérent.

**Exemple concret :**
```
"Résidu àll bi" — "Résidu" (français, non traduit) + "àll bi" (wolof)
```

**Cause probable.** Le modèle échoue à traduire complètement certains segments, probablement liés à la même difficulté que le point 2 (vocabulaire technique rare) — plutôt que de forcer une traduction incertaine, il semble parfois laisser le terme source tel quel, sans cohérence garantie sur quand il le fait ou non.

**Portée.** À distinguer du code-switching *naturel* et volontaire déjà observé dans le corpus KALLAAMA (S0) — celui-ci est un phénomène linguistique réel de l'usage oral du wolof (marqué `:fra` dans les annotations). Ce qu'on observe ici est différent : une **défaillance de traduction**, pas un choix linguistique authentique. Ne pas confondre les deux dans l'analyse.

**Statut.** À surveiller en relecture manuelle ; pas de correctif automatique identifié à ce stade.

---

## Synthèse — implication pour la suite

Ces trois limites confirment que NLLB-200 (version distillée 600M) reste **fragile sur le wolof**, cohérent avec son statut de langue à faibles ressources dans ce modèle. Aucune ne remet en cause l'usage de NLLB comme point de départ pour cette phase (l'objectif reste un premier jet, pas une traduction finale validée), mais toutes renforcent la nécessité déjà actée d'une **relecture humaine** avant indexation, et **a fortiori** d'une validation par un locuteur wolof natif (partenaires CRAC-UGB) avant tout usage en production.

**Point de vigilance pour S2-J2 (indexation ChromaDB wolof)** : si ces erreurs de traduction (confusions lexicales, mélanges de langue) se retrouvent dans les fiches indexées, elles pourraient dégrader la qualité du retrieval au-delà de ce qu'on a déjà observé en français (chevauchements lexicaux entre fiches proches, cf. diagnostic Q2/Q3 du 28 juillet) — un facteur supplémentaire, propre au wolof, à garder en tête lors du test critique des embeddings.

---

*Document à compléter au fil de la relecture des fiches restantes (S2-J1).*